# Pobreza monetaria en el Perú: ejecución del pipeline

Este cuaderno solo **ejecuta** el proyecto: toda la lógica está en el paquete `pobreza` (carpeta `src/`), organizado por capas y documentado en `docs/ARQUITECTURA.md`.

Orden de las etapas: descargar → construir → medir → entrenar → evaluar → reportar. Cada etapa guarda su salida, así que se puede repetir una sola.

## 1. Obtener el código

Opción A: desde GitHub (cambie `REPO_URL` por la dirección del repositorio).
Opción B: suba `pobreza-ml-peru.zip` a su Google Drive y deje `REPO_URL` vacío.

In [ ]:
REPO_URL = ""  # p. ej. "https://github.com/usuario/pobreza-ml-peru.git"
ZIP_EN_DRIVE = "/content/drive/MyDrive/pobreza-ml-peru.zip"

from google.colab import drive
drive.mount('/content/drive')

import os
if REPO_URL:
    !git clone -q {REPO_URL} /content/pobreza-ml-peru
else:
    !unzip -q -o {ZIP_EN_DRIVE} -d /content/
%cd /content/pobreza-ml-peru

In [ ]:
!pip install -q -e .

## 2. Descargar los microdatos del INEI

Baja 7 módulos por año (Sumaria y módulos 01, 02, 03, 04, 05 y 37) para 2017–2024. Los zips quedan en `datos/crudos/<año>/`. Si un archivo ya existe, no se vuelve a bajar.

Para guardar los zips entre sesiones, puede enlazar `datos/` a una carpeta de Drive (celda opcional).

In [ ]:
# Opcional: conservar los datos en Drive entre sesiones
CARPETA_DRIVE = "/content/drive/MyDrive/pobreza-ml-peru-datos"
os.makedirs(CARPETA_DRIVE, exist_ok=True)
if not os.path.islink("datos"):
    !rm -rf datos && ln -s {CARPETA_DRIVE} datos

In [ ]:
!pobreza descargar

## 3. Construir la base de hogares

Une los módulos por hogar (CONGLOME, VIVIENDA, HOGAR), apila los años y marca a los hogares entrevistados en más de un año.

In [ ]:
!pobreza construir
import pandas as pd
pd.read_csv("datos/procesados/resumen_panel.csv")

## 4. Medir la pobreza y validar contra el INEI

Si la tasa reproducida difiere de la oficial en más de 0,3 puntos, el pipeline se detiene. **No continúe si esta celda falla**: el modelo se entrenaría con un objetivo mal construido.

In [ ]:
!pobreza medir
pd.read_csv("resultados/reportes/datos/incidencia_anual.csv").round(2)

## 5. Entrenar y evaluar

Entrena XGBoost, bosque aleatorio, probit, logit y KNN en cada pliegue de la validación temporal, y la prueba final con 2024.

In [ ]:
!pobreza entrenar
!pobreza evaluar
pd.read_csv("resultados/reportes/datos/metricas_prueba.csv").round(4)

## 6. Reportes para la tesis y el dashboard

In [ ]:
!pobreza reportar
from IPython.display import Image, display
import glob
for ruta in sorted(glob.glob("resultados/reportes/figuras/*.png")):
    print(ruta); display(Image(ruta, width=700))

## 7. Guardar resultados en Drive

Copia figuras, tablas (`tablas_tesis.docx`), CSV, modelos y los JSON del dashboard.

In [ ]:
!mkdir -p /content/drive/MyDrive/pobreza-ml-peru-resultados
!cp -r resultados dashboard/public/data /content/drive/MyDrive/pobreza-ml-peru-resultados/
print("Listo")